In [ ]:
# input
tax_id_lineage_file = "./tmp/taxId-lineage.tsv"
afdb50_tax_id_file = "./tmp/afdb50_repId-entryId-taxId.tsv"
metaldb_tax_id_file = "./tmp/metaldb_repId-entryId-taxId.tsv"
# output

In [2]:
import pandas as pd

df_afdb50 = pd.read_table(afdb50_tax_id_file, header=None, names=["_", "__", "tax_id"], usecols=["tax_id"])
df_metaldb = pd.read_table(metaldb_tax_id_file, header=None, names=["_", "__", "tax_id"], usecols=["tax_id"])
df_tax_id = pd.read_table(tax_id_lineage_file, header=None, names=["tax_id", "lineage"])
df_tax_id = df_tax_id[df_tax_id['lineage'].notna()]
df_tax_id['superkingdom'] = df_tax_id['lineage'].map(lambda x: x.split("|")[0])
df_tax_id['kingdom'] = df_tax_id['lineage'].map(lambda x: x.split("|")[1])

In [3]:
df_metaldb = pd.merge(df_metaldb, df_tax_id, on="tax_id")
df_afdb50 = pd.merge(df_afdb50, df_tax_id, on="tax_id")
metaldb_lin2cnt = df_metaldb['lineage'].value_counts().to_dict()
afdb50_lin2cnt = df_afdb50['lineage'].value_counts().to_dict()

In [4]:
class Node:
    def __init__(self, parent, name: str):
        self.parent = parent
        if parent is not None:
            self.parent.children.append(self)
        self.name = name
        self.children = []
        self.cnt = 0
    
    def update_cnt(self, cnt: int):
        self.cnt += cnt
        p = self.parent
        while p is not None:
            p.cnt += cnt
            p = p.parent
    

def get_summary(info: dict):

    root = Node(parent=None, name="root")
    cellular = Node(parent=root, name="cellular organisms")
    others = Node(parent=root, name="others")
    names2node: dict[str, Node] = dict()
    domains = {'Bacteria', 'Eukaryota', 'Archaea'}
    for d in domains:
        n = Node(cellular, d)
        names2node[d] = n
    
    for k, v in info.items():
        items =  k.split("|")
        d, k = items
        if d in domains:
            d_n = names2node[d]
            k_n = Node(d_n, k)
            k_n.update_cnt(v)
        else:
            others.update_cnt(v)
    
    records = []
    q: list[Node] = root.children[:]
    while q:
        cur = q.pop()
        records.append({
            "src": cur.parent.name,
            "tgt": cur.name,
            "cnt": cur.cnt,
            "ratio": cur.cnt / sum([i.cnt for i in cur.parent.children])
        })
        for c in cur.children:
            q.append(c)

    return pd.DataFrame(records)

In [5]:
df = pd.merge(get_summary(metaldb_lin2cnt), get_summary(afdb50_lin2cnt), on=['src', 'tgt'], suffixes=("_metaldb", "_afdb50"))
df = df.sort_values(by=['src', 'tgt'])
df

,src,tgt,cnt_metaldb,ratio_metaldb,cnt_afdb50,ratio_afdb50
3,Archaea,,51718,0.037469,274889,0.047015
7,Archaea,Methanobacteriati,829226,0.600768,3460221,0.591817
5,Archaea,Nanobdellati,83263,0.060323,388881,0.066512
4,Archaea,Promethearchaeati,65934,0.047769,244995,0.041903
6,Archaea,Thermoproteati,350136,0.253671,1477791,0.252753
14,Bacteria,,6840998,0.249459,34203096,0.226790
15,Bacteria,Bacillati,10117956,0.368955,55459066,0.367732
16,Bacteria,Pseudomonadati,10464333,0.381586,61151676,0.405478
9,Eukaryota,,707142,0.077134,5086658,0.091477
11,Eukaryota,Fungi,2159820,0.235589,12367717,0.222418


In [6]:
for _, row in df.iterrows():
    src, tgt = row['src'], row['tgt']
    if tgt != "":
        cnt = row['cnt_metaldb']
        print(f"{src}[{cnt}]{tgt}")
# https://github.com/nowthis/sankeymatic for draw sankey fig

Archaea[829226]Methanobacteriati
Archaea[83263]Nanobdellati
Archaea[65934]Promethearchaeati
Archaea[350136]Thermoproteati
Bacteria[10117956]Bacillati
Bacteria[10464333]Pseudomonadati
Eukaryota[2159820]Fungi
Eukaryota[4440444]Metazoa
Eukaryota[1860354]Viridiplantae
cellular organisms[1380277]Archaea
cellular organisms[27423287]Bacteria
cellular organisms[9167760]Eukaryota
root[37971324]cellular organisms
root[389659]others
